# SchemeXpress — NLP & TF-IDF Similarity

This notebook demonstrates the classical NLP pipeline used by SchemeXpress
to retrieve government schemes relevant to a user's natural-language requirement.

Pipeline:

Cleaned Dataset
→ Text Preprocessing
→ Combined Scheme Text
→ TF-IDF Vectorization
→ Cosine Similarity
→ Top-K Scheme Retrieval

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from src.nlp import normalize_text


In [3]:
print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

NameError: name 'sys' is not defined

In [3]:
from src.nlp import normalize_text

print(normalize_text("Financial Assistance for Higher Education!"))

ModuleNotFoundError: No module named 'src'

In [4]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\asus\Documents\SchemeXpress


In [5]:
from src.nlp import normalize_text

print(normalize_text("Financial Assistance for Higher Education!"))

financial assistance for higher education


In [6]:
DATA_PATH = Path("../data/processed/cleaned_schemes.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

NameError: name 'pd' is not defined

In [7]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from src.nlp import normalize_text

print("Imports successful")

Imports successful


In [9]:
DATA_PATH = Path("../data/processed/cleaned_schemes.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (3397, 11)

Columns:
['scheme_name', 'slug', 'details', 'benefits', 'eligibility', 'application', 'documents', 'level', 'schemeCategory', 'tags', 'combined_text']


In [10]:
text_columns = [
    column
    for column in [
        "scheme_name",
        "details",
        "benefits",
        "eligibility",
        "application",
        "documents",
        "tags",
        "combined_text",
    ]
    if column in df.columns
]

print("Available text fields:")
for column in text_columns:
    print("-", column)

Available text fields:
- scheme_name
- details
- benefits
- eligibility
- application
- documents
- tags
- combined_text


In [11]:
df[["scheme_name", "combined_text"]].head(3)

,scheme_name,combined_text
0,"Immediate Relief Assistance"" under ""Welfare an...","Immediate Relief Assistance"" under ""Welfare an..."
1,AICTE SHORT TERM TRAINING PROGRAMME-SFURTI SCHEME,AICTE SHORT TERM TRAINING PROGRAMME-SFURTI SCH...
2,Burial and Ex-gratia Payment Scheme in Case of...,Burial and Ex-gratia Payment Scheme in Case of...


In [12]:
df["nlp_text"] = df["combined_text"].fillna("").map(normalize_text)

print(df["nlp_text"].head(3).to_string())

0    immediate relief assistance under welfare and ...
1    aicte short term training programme sfurti sch...
2    burial and ex gratia payment scheme in case of...


In [13]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

tfidf_matrix = vectorizer.fit_transform(df["nlp_text"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

TF-IDF matrix shape: (3397, 10000)
Vocabulary size: 10000


In [14]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

tfidf_matrix = vectorizer.fit_transform(df["nlp_text"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

TF-IDF matrix shape: (3397, 10000)
Vocabulary size: 10000


In [15]:
feature_names = vectorizer.get_feature_names_out()

print("First 50 vocabulary terms:")
print(feature_names[:50])

First 50 vocabulary terms:
['00' '00 00' '00 000' '00 ha' '00 lakh' '00 lakhs' '000' '000 00'
 '000 000' '000 10' '000 15' '000 20' '000 25' '000 40' '000 50' '000 60'
 '000 75' '000 and' '000 are' '000 as' '000 at' '000 each' '000 financial'
 '000 for' '000 group' '000 if' '000 in' '000 interest' '000 is'
 '000 lump' '000 maximum' '000 note' '000 on' '000 one' '000 only'
 '000 or' '000 other' '000 per' '000 rupees' '000 shall' '000 subsidy'
 '000 the' '000 to' '000 total' '000 whichever' '000 will' '000 with'
 '001' '01' '01 01']


In [16]:
print("Total vocabulary terms:", len(feature_names))

Total vocabulary terms: 10000


In [17]:
user_query = "financial assistance for higher education"

query_text = normalize_text(user_query)

query_vector = vectorizer.transform([query_text])

print("User query:", user_query)
print("Query vector shape:", query_vector.shape)

User query: financial assistance for higher education
Query vector shape: (1, 10000)


In [18]:
similarity_scores = cosine_similarity(
    query_vector,
    tfidf_matrix
).flatten()

print("Number of similarity scores:", len(similarity_scores))
print("Highest similarity:", similarity_scores.max())
print("Lowest similarity:", similarity_scores.min())

Number of similarity scores: 3397
Highest similarity: 0.2659416079819347
Lowest similarity: 0.0


In [19]:
top_k = 10

top_indices = np.argsort(similarity_scores)[::-1][:top_k]

results = df.iloc[top_indices].copy()

results["similarity_score"] = similarity_scores[top_indices]

results[["scheme_name", "similarity_score"]]

,scheme_name,similarity_score
850,Economic Help To Tribal Girls For Higher Educa...,0.265942
851,Economic Help To Tribal Girls For Higher Educa...,0.240755
3291,Vikramaditya Yojna,0.225168
1167,Gaon Ki Beti,0.212292
655,Dayanand Bandodkar Scheme For Higher Education...,0.205218
3267,Vahli Dikri Yojana,0.187729
577,Child Benefit Scheme (UKBOCWWB),0.182978
2864,Scholarship Scheme For Disabled Students To St...,0.181392
1105,Free Books Scheme for Scheduled Caste Students...,0.170456
706,Devnarayan Scooty Distribution And Incentive S...,0.167665


In [20]:
results[
    ["scheme_name", "similarity_score"]
].reset_index(drop=True)

,scheme_name,similarity_score
0,Economic Help To Tribal Girls For Higher Educa...,0.265942
1,Economic Help To Tribal Girls For Higher Educa...,0.240755
2,Vikramaditya Yojna,0.225168
3,Gaon Ki Beti,0.212292
4,Dayanand Bandodkar Scheme For Higher Education...,0.205218
5,Vahli Dikri Yojana,0.187729
6,Child Benefit Scheme (UKBOCWWB),0.182978
7,Scholarship Scheme For Disabled Students To St...,0.181392
8,Free Books Scheme for Scheduled Caste Students...,0.170456
9,Devnarayan Scooty Distribution And Incentive S...,0.167665


In [21]:
test_queries = [
    "support for farmers and agriculture",
    "women entrepreneurship financial support",
    "healthcare assistance for poor families",
    "scholarship for students",
]

for query in test_queries:
    query_vector = vectorizer.transform([normalize_text(query)])

    scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    indices = np.argsort(scores)[::-1][:5]

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    for rank, index in enumerate(indices, start=1):
        print(
            f"{rank}. {df.iloc[index]['scheme_name']} "
            f"(similarity={scores[index]:.4f})"
        )
    


QUERY: support for farmers and agriculture
1. Farmers Training Institute Scheme (similarity=0.2181)
2. Soil Testing Laboratory (similarity=0.2020)
3. Atma Nirbhar Krishi Yojana (similarity=0.1980)
4. Agricultural Skill Development Training Programme For Women Farmers And Farmers (similarity=0.1902)
5. Kisan Kaleva Yojana (similarity=0.1755)

QUERY: women entrepreneurship financial support
1. Deen Dayal Upadhyaya Bunkar Yojana (similarity=0.1556)
2. Savitribai Phule Self-Help Scheme (similarity=0.1331)
3. One Time Financial Support for Economically Weaker Meritorious ST Students (similarity=0.1290)
4. Rehabilitation of Cured Leprosy Persons: Marriage Incentives & Support (similarity=0.1280)
5. Financial Support Scheme (similarity=0.1267)

QUERY: healthcare assistance for poor families
1. Marriage Grant Scheme (similarity=0.1963)
2. Scheme for Establishment of Backyard Poultry Units (similarity=0.1742)
3. Mukhya Mantri Grihini Suvidha Yojana (similarity=0.1596)
4. Tripura Health Assuran

## Interpretation

The TF-IDF and cosine similarity pipeline successfully converts
government scheme descriptions and user queries into numerical vectors
and retrieves schemes based on textual similarity.

The test queries demonstrate that the method can identify strongly
related schemes for topics such as agriculture and scholarships.

However, TF-IDF is a lexical retrieval technique. It primarily relies
on vocabulary overlap and does not understand user intent or verify
eligibility conditions. Some queries therefore produce less relevant
rankings.

For this reason, TF-IDF will be used as a candidate retrieval component
rather than the complete recommendation engine.

The next stage will introduce eligibility filtering and a hybrid
ranking approach.

## Current Limitations

1. TF-IDF relies heavily on word overlap.
2. Similarity scores are not eligibility probabilities.
3. Text similarity alone cannot verify whether a user qualifies.
4. Different wording can describe the same concept but receive a lower score.
5. Highly overlapping words can produce misleading matches.
6. The current system does not yet use structured user attributes.
7. The current system does not yet implement hybrid recommendation scoring.